# GBM minimal demo walkthrough

End-to-end tour of the exact pair-level co-occurrence mining pipeline on a small
synthetic presence/absence matrix. Build the engines first (see `Dockerfile`,
`build_gbm.bat`, or the CI workflow), then set `BIN` below to the executable directory.
Everything here is deterministic: re-running a cell gives byte-identical output.

This notebook is the interactive tour: it generates its own synthetic data so you can
see how matrices and planted benchmarks are made. If you just want a one-command,
self-checking run of the pipeline on a bundled dataset, use `demo/run_demo.sh` (or
`demo\run_demo.bat`) instead.

In [ ]:
import os, subprocess, sys, tempfile
BIN = os.path.join('..', 'src', 'gbm')   # adjust if your binaries live elsewhere
if os.name == 'nt':
    EXE = {n: os.path.join(BIN, n + '.exe') for n in ('gbm_count', 'gbm_stats', 'gbm_sim')}
else:
    EXE = {n: os.path.join(BIN, n) for n in ('gbm_count', 'gbm_stats', 'gbm_sim')}
tmp = tempfile.mkdtemp(prefix='gbm_demo_')
def run(exe, args):
    r = subprocess.run([exe] + args, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    assert r.returncode == 0
    return r
print('work dir:', tmp)

## 1. Generate (or load) a GBM1 presence/absence matrix

Format: first line `GBM1 <N genomes> <M gene families>`, then one line per family
`<name> <N-bit 0/1 string>`. Real matrices are converted from public pangenome
resources with `src/gbm/h5_to_gbm1.py` (see README, Data availability). Here we
generate a synthetic one (N=512 genomes, M=2,000 families, density 0.1).

In [ ]:
matrix = os.path.join(tmp, 'demo.txt')
run(EXE['gbm_count'], ['--gen', '512', '2000', '42', matrix])
print(open(matrix).readline().strip(), '...')

## 2. All-pairs exact counting

`--count <in> <out> <minc> [dep|screen]` writes one row per tested pair:
`famA famB c11 supA supB c10 c01 c00`. `minc=5` keeps enrichment pairs with
c11 >= 5; `dep` additionally keeps depletion pairs (c11 = 0) whose independent
expectation supA*supB/N is at least minc.

In [ ]:
pairs = os.path.join(tmp, 'pairs.tsv')
run(EXE['gbm_count'], ['--count', matrix, pairs, '5', 'dep'])
with open(pairs) as f:
    header = f.readline().strip()
    n = sum(1 for _ in f)
print(header)
print('tested pairs:', n)

## 3. Pair-level statistics

`gbm_stats --pairs` computes two-sided Fisher exact p-values (R-compatible
definition) and odds ratios for every counted pair.

In [ ]:
stats = os.path.join(tmp, 'stats.tsv')
run(EXE['gbm_stats'], ['--pairs', pairs, stats])
with open(stats) as f:
    print(f.readline().strip())
    for i, line in zip(range(3), f):
        print(line.strip())

## 4. Planted ground truth

`gbm_sim --plant` clears two fresh families and plants a known interaction
(co-occurrence / mutual exclusion / conditional) into the matrix, writing the
truth table it used. This is the synthetic equivalent of the study's planted
benchmark and lets anyone validate recall statistics on public-free data.

In [ ]:
planted = os.path.join(tmp, 'planted.txt')
truth = os.path.join(tmp, 'truth.tsv')
run(EXE['gbm_sim'], ['--plant', matrix, planted, truth, '10', '10', '10', '7'])
print(open(truth).read())

## 5. Verify recoverability

Re-count the planted matrix and check each planted co-occurrence pair surfaces
with its target support (and each planted exclusion pair is nowhere enriched).
The same checks run automatically in `tests/smoke_test.py`.

In [ ]:
planted_pairs = os.path.join(tmp, 'planted_pairs.tsv')
run(EXE['gbm_count'], ['--count', planted, planted_pairs, '2', 'dep'])
got = {}
with open(planted_pairs) as f:
    f.readline()
    for line in f:
        t = line.split('\t')
        got[(t[0], t[1])] = int(t[2])
ok = 0
with open(truth) as f:
    f.readline()
    for line in f:
        typ, a, b, ts, _ = line.rstrip('\n').split('\t')
        if typ == 'cooc' and got.get((a, b), got.get((b, a))) == int(ts):
            ok += 1
print(f'planted co-occurrence pairs recovered: {ok}/10')